# Engine PINN End-to-End Notebook

This notebook combines the full workflow:
- Data loading (or synthetic fallback)
- Data normalization and dataloaders
- Physics-Informed Neural Network (PINN) construction
- Two-phase training (latent pretrain + full training)
- Evaluation and latent analysis
- Plot generation and artifact export


In [ ]:
from pathlib import Path
import logging
import random

import numpy as np
import pandas as pd
import torch

from engine_pinn.data.dataset import build_dataloaders, load_or_generate_dataframe
from engine_pinn.models.pinn import EnginePINN
from engine_pinn.training.loss import PINNLoss
from engine_pinn.training.trainer import Trainer
from engine_pinn.utils.config import PhysicsConfig, TrainConfig
from engine_pinn.utils.plotting import plot_latent_trends, plot_training_curves


In [ ]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

logging.basicConfig(level=logging.INFO, format='%(asctime)s | %(levelname)s | %(name)s | %(message)s')
logger = logging.getLogger('notebook')

train_cfg = TrainConfig()
phys_cfg = PhysicsConfig()
set_seed(train_cfg.seed)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device


## 1) Load Data (CSV if present, otherwise synthetic)


In [ ]:
df = load_or_generate_dataframe(train_cfg.data_csv)
print(df.head())
print('Rows:', len(df))


In [ ]:
bundle = build_dataloaders(
    df,
    batch_size=train_cfg.batch_size,
    test_size=train_cfg.test_size,
    val_size=train_cfg.val_size,
    random_state=train_cfg.seed,
)
len(bundle.train_loader.dataset), len(bundle.val_loader.dataset), len(bundle.test_loader.dataset)


## 2) Build PINN Model
Physics layer in the model computes:
- BSFC = (3600 / LHV) * ((BMEP + L1) / (BMEP * L2))
- NOx = A * exp(-B / L3)


In [ ]:
nox_a_init = max(float(df['NOx'].max()), 1e-3)
model = EnginePINN(
    lhv=phys_cfg.lhv,
    nox_a_init=nox_a_init,
    nox_b_init=phys_cfg.nox_b_init,
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=train_cfg.lr, weight_decay=train_cfg.weight_decay)
loss_fn = PINNLoss(lambda_phys=train_cfg.lambda_phys, lambda_mono=train_cfg.lambda_mono)
trainer = Trainer(
    model=model,
    loss_fn=loss_fn,
    optimizer=optimizer,
    device=device,
    checkpoint_dir=train_cfg.checkpoint_dir,
    patience=train_cfg.patience,
)
model


## 3) Train (Two Phases)
- Phase 1: freeze NOx physics scalars A/B
- Phase 2: unfreeze all parameters


In [ ]:
history = trainer.fit(
    bundle.train_loader,
    bundle.val_loader,
    epochs_phase1=train_cfg.epochs_phase1,
    epochs_phase2=train_cfg.epochs_phase2,
)
print('Training epochs recorded:', len(history.train_total))


## 4) Evaluate and Collect Latent Variables


In [ ]:
def evaluate_and_collect_latents(model, loader, y_scaler, device):
    model.eval()
    rows = []
    with torch.no_grad():
        for batch in loader:
            x = batch['x'].to(device)
            x_raw = batch['x_raw'].to(device)
            out = model(x, x_raw)

            y_true = y_scaler.inverse_transform(batch['y'].numpy())
            y_pred = np.hstack([out['bsfc'].cpu().numpy(), out['nox'].cpu().numpy()])
            latent = out['latents'].cpu().numpy()
            x_raw_np = batch['x_raw'].cpu().numpy()

            for i in range(len(x_raw_np)):
                rows.append({
                    'BMEP': x_raw_np[i, 0],
                    'H2_percentage': x_raw_np[i, 1],
                    'Spark_Ignition_Timing': x_raw_np[i, 2],
                    'Lambda': x_raw_np[i, 3],
                    'BSFC_true': y_true[i, 0],
                    'NOx_true': y_true[i, 1],
                    'BSFC_pred': y_pred[i, 0],
                    'NOx_pred': y_pred[i, 1],
                    'L1_FMEP': latent[i, 0],
                    'L2_Indicated_Efficiency': latent[i, 1],
                    'L3_Temp_Potential': latent[i, 2],
                })
    return pd.DataFrame(rows)

results_df = evaluate_and_collect_latents(model, bundle.test_loader, bundle.y_scaler, device)
results_df.head()


## 5) Plot Convergence and Sensitivity Analysis


In [ ]:
train_cfg.plots_dir.mkdir(parents=True, exist_ok=True)

training_curve_path = train_cfg.plots_dir / 'training_curves.png'
latent_plot_path = train_cfg.plots_dir / 'latent_sensitivity.png'
results_path = train_cfg.plots_dir / 'test_predictions_with_latents.csv'

plot_training_curves({'train_total': history.train_total, 'val_total': history.val_total}, training_curve_path)
plot_latent_trends(results_df, latent_plot_path)
results_df.to_csv(results_path, index=False)

print('Saved:', training_curve_path)
print('Saved:', latent_plot_path)
print('Saved:', results_path)


## 6) Quick Diagnostics


In [ ]:
summary = results_df[['BSFC_true', 'BSFC_pred', 'NOx_true', 'NOx_pred']].describe().T
summary
